# Clean 1,000-audio Whisper LoRA training on Kaggle
This notebook trains LoRA adapters on the cleaned Tunisian subset: 889 train examples and 111 validation examples. The baseline model and tokenizer are preserved. The external 96-utterance test is never loaded.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/mehdiyds/tunisian-speech-ai.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'codex/smoke-test-whisper-fixes')
REPO_DIR = Path('/kaggle/working/tunisian-speech-ai')
if not (REPO_DIR / 'scripts' / 'train_lora.py').exists():
    if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
required = [REPO_DIR / 'scripts' / 'train_lora.py', REPO_DIR / 'src' / 'data' / 'validate_dataset.py']
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing repository files: ' + ', '.join(missing))
sys.path.insert(0, str(REPO_DIR))
print('Repository:', REPO_DIR)
print('Branch:', REPO_BRANCH)

In [ ]:
%pip install -q --upgrade --force-reinstall --no-deps transformers==4.48.3 tokenizers==0.21.0 huggingface-hub==0.27.1 accelerate==1.2.1 peft==0.14.0 datasets==3.2.0 evaluate==0.4.3 jiwer==3.0.5 rapidfuzz==3.11.0 soundfile==0.13.0 PyYAML==6.0.2
print('Dependencies installed. Restart the kernel if Kaggle requests it.')

In [ ]:
import torch
if not hasattr(torch, '_utils'): raise RuntimeError('PyTorch is corrupted. Use Kaggle Factory reset; never install requirements.txt wholesale.')
if not torch.cuda.is_available(): raise RuntimeError('CUDA unavailable. Enable a Kaggle GPU.')
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

In [ ]:
DATASET_CANDIDATES = [Path('/kaggle/input/tundataset/data')]
DATASET_DIR = next((p for p in DATASET_CANDIDATES if (p / 'splits' / 'train.jsonl').exists()), None)
if DATASET_DIR is None:
    found = list(Path('/kaggle/input').glob('**/data/splits/train.jsonl'))
    DATASET_DIR = found[0].parent.parent if found else None
if DATASET_DIR is None: raise FileNotFoundError('train.jsonl not found under /kaggle/input')
AUDIO_DIR = DATASET_DIR / 'processed' / 'audio'
SOURCE_TRAIN = DATASET_DIR / 'splits' / 'train.jsonl'
SOURCE_VALIDATION = DATASET_DIR / 'splits' / 'validation.jsonl'
if not AUDIO_DIR.exists(): raise FileNotFoundError(f'Audio directory not found: {AUDIO_DIR}')
def absolute_manifest(source, name):
    records = []
    for line in source.read_text(encoding='utf-8-sig').splitlines():
        if not line.strip(): continue
        record = json.loads(line)
        audio = Path(record['audio'])
        if audio.parts and audio.parts[0] == 'data': audio = Path(*audio.parts[1:])
        record['audio'] = str(DATASET_DIR / audio)
        if not Path(record['audio']).exists(): raise FileNotFoundError(record['audio'])
        records.append(record)
    output = Path('/kaggle/working') / name
    output.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in records), encoding='utf-8')
    return output
TRAIN_MANIFEST = absolute_manifest(SOURCE_TRAIN, 'clean_1000_train_absolute.jsonl')
VALIDATION_MANIFEST = absolute_manifest(SOURCE_VALIDATION, 'clean_1000_validation_absolute.jsonl')
print('Dataset:', DATASET_DIR)
print('Audio files:', len(list(AUDIO_DIR.glob('*.wav'))))
print('Train records: 889 expected')
print('Validation records: 111 expected')

In [ ]:
validate = REPO_DIR / 'src' / 'data' / 'validate_dataset.py'
subprocess.run([sys.executable, str(validate), str(TRAIN_MANIFEST)], check=True)
subprocess.run([sys.executable, str(validate), str(VALIDATION_MANIFEST)], check=True)
print('Both manifests validated successfully.')

## Clean 1,000-audio training
The configuration remains deliberately conservative: LoRA rank 8, q_proj/v_proj, batch size 1, accumulation 8, fp16, gradient checkpointing, learning rate 1e-4, linear scheduler, and 5 epochs. Batch size 1 is kept for GPU memory safety.

This run uses only the cleaned subset. Do not use the external 96-utterance benchmark here.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/lora_clean_1000')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', str(REPO_DIR / 'scripts' / 'train_lora.py'), '--train-manifest', str(TRAIN_MANIFEST), '--validation-manifest', str(VALIDATION_MANIFEST), '--output-dir', str(OUTPUT_DIR), '--device', 'cuda', '--logging-steps', '1', '--eval-steps', '100', '--save-steps', '100']
print('CLEAN 1,000-AUDIO TRAINING COMMAND:')
print(' '.join(command))
print('Train: 889 clean examples | Validation: 111 clean examples | Epochs: 5 | Device batch size: 1')
print('STATUS: phase messages are shown immediately; training loss is printed after every optimizer update (8 audios), and validation/checkpoints occur every 100 updates.')
started = time.time()
completed = subprocess.run(command, check=False, env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
minutes = (time.time() - started) / 60
print(f'STATUS: process finished with exit code {completed.returncode} after {minutes:.1f} minutes.')
if completed.returncode != 0: raise subprocess.CalledProcessError(completed.returncode, command)

In [ ]:
import shutil
metrics = OUTPUT_DIR / 'metrics.json'
if metrics.exists(): print(metrics.read_text(encoding='utf-8'))
files = [p for p in sorted(OUTPUT_DIR.rglob('*')) if p.is_file()]
print('Output files:', len(files))
for path in files[:30]: print(path.relative_to(OUTPUT_DIR))
archive = shutil.make_archive('/kaggle/working/lora_clean_1000_outputs', 'zip', OUTPUT_DIR)
print('Download archive:', archive)

## Resume after interruption
Keep `/kaggle/working/lora_clean_1000/checkpoints/checkpoint-N` and rerun the training command with:

```python
command += ['--resume-from-checkpoint', '/kaggle/working/lora_clean_1000/checkpoints/checkpoint-N']
subprocess.run(command, check=True)
```

Download `lora_clean_1000_outputs.zip` before ending the Kaggle session.